In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv("data/students.csv")
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [ ]:
X = df.drop("math_score", axis=1)
y = df["math_score"]

In [5]:
X

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75
...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,99,95
996,male,group C,high school,free/reduced,none,55,55
997,female,group C,high school,free/reduced,completed,71,65
998,female,group D,some college,standard,completed,78,77


In [6]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [ ]:
numerical_features = X.select_dtypes(exclude="object").columns
categorical_features = X.select_dtypes(include="object").columns

C:\Users\Shreyansh_2\AppData\Local\Temp\ipykernel_4216\2647891575.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include='object').columns


In [8]:
numerical_features

Index(['reading_score', 'writing_score'], dtype='str')

In [9]:
categorical_features

Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course'],
      dtype='str')

In [ ]:
ohe = OneHotEncoder(drop="first")
scaler = StandardScaler()

In [ ]:
preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", ohe, categorical_features),
        ("StandardScaler", scaler, numerical_features),
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [13]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [14]:
X_train.shape

(750, 14)

In [15]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_scr = r2_score(true, predicted)

    return mae, mse, rmse, r2_scr

In [20]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "AdaBoostRegressor": AdaBoostRegressor(),
    "CatBoostRegressor": CatBoostRegressor(),
    "SVR": SVR()
}

models_list = []
r2_list = []

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    ## make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Evaluate train and test datasets
    train_mae, train_mse, train_rmse, train_r2_scr = evaluate_model(
        y_train, y_train_pred
    )
    test_mae, test_mse, test_rmse, test_r2_scr = evaluate_model(y_test, y_test_pred)

    print(list(models.values())[i])
    models_list.append(list(models.keys())[i])

    print("Model performance for Training set")
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2_scr))

    print("----------------------------------")

    print("Model performance for Test set")
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2_scr))
    r2_list.append(test_r2_scr)

    print("=" * 35)
    print("\n")


LinearRegression()
Model performance for Training set
- Root Mean Squared Error: 5.2972
- Mean Absolute Error: 4.2383
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.4825
- Mean Absolute Error: 4.3379
- R2 Score: 0.8778


Ridge()
Model performance for Training set
- Root Mean Squared Error: 5.2977
- Mean Absolute Error: 4.2371
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.4814
- Mean Absolute Error: 4.3365
- R2 Score: 0.8779


Lasso()
Model performance for Training set
- Root Mean Squared Error: 6.5469
- Mean Absolute Error: 5.1796
- R2 Score: 0.8080
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.6501
- Mean Absolute Error: 5.2184
- R2 Score: 0.8202


KNeighborsRegressor()
Model performance for Training set
- Root Mean Squared Error: 5.6662
- Mean Absolute Error: 4.5136
- R2 Score: 0.8562
------------------

In [21]:
for model, scr in zip(models_list, r2_list):
    print(model, scr)

LinearRegression 0.8778243107659013
Ridge 0.877873971198214
Lasso 0.8202480287428798
KNeighborsRegressor 0.7901646026811986
DecisionTreeRegressor 0.7436669677541955
RandomForestRegressor 0.847850585069027
XGBRegressor 0.8341626524925232
AdaBoostRegressor 0.8446565687845835
CatBoostRegressor 0.8536881705730431
SVR 0.7120023175761258
